# Sentiment Analysis of X (Twitter) Data Using NLP

**Dataset:** `X data.csv`  
**Task:** Classify X posts as Positive, Negative, or Neutral and identify sentiment patterns.

### Research Questions
1. What is the distribution of positive, neutral, and negative posts?
2. Which words are most strongly associated with each sentiment class?
3. How does text length vary across sentiment categories?
4. Can an NLP classifier accurately predict the supplied sentiment labels?
5. What patterns and limitations are visible in this dataset?

> **Important limitation:** the supplied CSV contains no timestamp/date column, so genuine sentiment-over-time analysis cannot be performed from this file alone.


## 1. Import libraries and load the dataset

In [ ]:
# If running in Google Colab, upload X data.csv first.
# !pip -q install wordcloud scikit-learn seaborn

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

DATA_PATH = "X data.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
display(df.head())


## 2. Data quality check

In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("missing"))

print("Duplicate rows:", df.duplicated().sum())

print("\nOriginal category distribution:")
display(df["category"].value_counts(dropna=False).sort_index())


## 3. Cleaning and sentiment labels

The supplied `category` field contains three values: `-1`, `0`, and `1`. For this project they are interpreted as **Negative, Neutral, and Positive**, respectively. The mapping should be checked against the original dataset documentation if available.


In [ ]:
# Remove rows without text or sentiment label
df = df.dropna(subset=["clean_text", "category"]).copy()

# Normalize labels
df["category"] = df["category"].astype(int)

label_map = {-1: "Negative", 0: "Neutral", 1: "Positive"}
df["sentiment"] = df["category"].map(label_map)

# Remove any unexpected labels
df = df.dropna(subset=["sentiment"]).copy()

# Remove duplicate posts
df = df.drop_duplicates(subset=["clean_text"]).copy()

print("Final rows:", len(df))
display(df["sentiment"].value_counts().reindex(["Positive","Neutral","Negative"]).to_frame("count"))


## 4. Basic exploratory analysis

In [ ]:
df["word_count"] = df["clean_text"].astype(str).str.split().str.len()
df["char_count"] = df["clean_text"].astype(str).str.len()

summary = (
    df["sentiment"]
    .value_counts()
    .reindex(["Positive", "Neutral", "Negative"])
    .to_frame("Count")
)
summary["Percentage"] = (summary["Count"] / len(df) * 100).round(2)

display(summary)
display(df[["word_count","char_count"]].describe().round(2))


## 5. Visualization: sentiment distribution

In [ ]:
plt.figure(figsize=(8,5))
order = ["Positive", "Neutral", "Negative"]
sns.countplot(data=df, x="sentiment", order=order)
plt.title("Overall Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Posts")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7,7))
counts = df["sentiment"].value_counts().reindex(order)
plt.pie(counts, labels=counts.index, autopct="%1.1f%%", startangle=90)
plt.title("Percentage Distribution of Sentiments")
plt.tight_layout()
plt.show()


## 6. Most frequent words

In [ ]:
# Remove a small set of generic stop words for frequency analysis
stop_words = set([
    "the","to","and","of","a","in","is","for","on","that","this","it","with",
    "was","are","be","as","at","by","from","or","an","have","has","had","not",
    "but","they","you","we","he","she","i","my","our","their","will","would",
    "can","could","should","just","all","about","what","who","how","when","why"
])

all_words = []
for text in df["clean_text"].astype(str):
    all_words.extend(w for w in re.findall(r"\b[a-zA-Z]{2,}\b", text.lower())
                     if w not in stop_words)

freq = Counter(all_words)
top20 = pd.DataFrame(freq.most_common(20), columns=["word","frequency"])
display(top20)

plt.figure(figsize=(10,6))
sns.barplot(data=top20, x="frequency", y="word")
plt.title("Top 20 Most Frequent Words")
plt.xlabel("Frequency")
plt.ylabel("Word")
plt.tight_layout()
plt.show()


## 7. Sentiment-specific word clouds

In [ ]:
for sentiment in order:
    text = " ".join(df.loc[df["sentiment"] == sentiment, "clean_text"].astype(str))
    wc = WordCloud(width=1200, height=600, background_color="white",
                   stopwords=stop_words, max_words=100).generate(text)
    plt.figure(figsize=(12,6))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"{sentiment} Sentiment Word Cloud")
    plt.show()


## 8. Text-length analysis

In [ ]:
plt.figure(figsize=(9,5))
sns.boxplot(data=df, x="sentiment", y="word_count", order=order)
plt.title("Tweet Length by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Word Count")
plt.ylim(0, df["word_count"].quantile(0.99))
plt.tight_layout()
plt.show()

display(
    df.groupby("sentiment")[["word_count","char_count"]]
      .agg(["count","mean","median","std"])
      .round(2)
)


## 9. NLP classification using TF-IDF + Logistic Regression

The dataset already supplies sentiment labels. To demonstrate NLP-based categorization independently, a supervised classifier is trained using TF-IDF text features and Logistic Regression.

**Evaluation:** accuracy, precision, recall, F1-score, and confusion matrix.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"].astype(str),
    df["sentiment"],
    test_size=0.20,
    random_state=42,
    stratify=df["sentiment"]
)

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=3,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training matrix:", X_train_tfidf.shape)
print("Testing matrix:", X_test_tfidf.shape)

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))


## 10. Confusion matrix

In [ ]:
labels = ["Negative", "Neutral", "Positive"]
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix — TF-IDF + Logistic Regression")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()


## 11. Most influential words for each class

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = model.coef_

class_names = model.classes_
for i, cls in enumerate(class_names):
    top_positive = np.argsort(coefs[i])[-15:][::-1]
    print(f"\nTop terms associated with {cls}:")
    print(", ".join(feature_names[top_positive]))


## 12. Statistical analysis

In [ ]:
# Descriptive statistics for text length
stats = (
    df.groupby("sentiment")["word_count"]
      .agg(["count","mean","median","std","min","max"])
      .reindex(order)
      .round(2)
)
display(stats)

# Chi-square goodness-of-fit against equal class proportions
from scipy.stats import chisquare

observed = summary["Count"].values
expected = np.repeat(observed.sum()/3, 3)
chi2, p = chisquare(observed, f_exp=expected)

print("Chi-square goodness-of-fit test against equal class proportions")
print("Chi-square statistic:", round(chi2, 3))
print("p-value:", p)


## 13. Research findings template

After running all cells, summarize the actual numerical results:

- **RQ1:** Report the counts and percentages of positive, neutral, and negative posts.
- **RQ2:** Since this CSV has no date/time field, state that temporal sentiment trends cannot be measured from this dataset.
- **RQ3:** Report the most frequent words and sentiment-specific terms.
- **RQ4:** Compare average/median text length across sentiment classes.
- **RQ5:** Report the classifier accuracy and class-wise precision, recall, and F1-score.

### Interpretation caution
The dataset is a labeled sample of X posts and should not be treated as a complete measurement of all X users' opinions. Automated classification can also struggle with sarcasm, irony, slang, context, multilingual content, and ambiguous statements.


## 14. Conclusion

This project applies natural language processing and statistical analysis to a labeled dataset of X posts. The workflow includes data cleaning, sentiment distribution analysis, word-frequency analysis, sentiment-specific visualization, TF-IDF feature extraction, supervised sentiment classification, and model evaluation.

The findings describe patterns within the supplied dataset. They should not be generalized to all X users without a representative sampling design. A timestamped dataset would be required for a valid sentiment-over-time analysis.


## 15. References

1. NLTK VADER documentation: https://www.nltk.org/
2. scikit-learn documentation: https://scikit-learn.org/
3. X API documentation: https://docs.x.com/
4. Dataset/source documentation should be added here if your instructor provided the original dataset source.
